In [ ]:
# ── Run this chapter on a clean machine (Colab "Julia" runtime, Binder, or any Jupyter with a Julia kernel) ──
# Cell 1 of 2 — the engine and the data. Measured on a clean machine: about three minutes to a first fit.
# Both engines are public on GitHub, so nothing needs a registry.
import Pkg
Pkg.add(url = "https://github.com/itchyshin/DRM.jl", rev = "26f4c4ddca58bd672fce807a7564face3debdc93")   # the commit the chapters were executed against
Pkg.add(["DataFrames", "CSV", "Distributions", "StatsBase", "StatsModels"])
REPO_RAW = "https://raw.githubusercontent.com/itchyshin/stats-hours/main"   # works once the repository is public
for f in ("tools/theme_itchy.jl", "tools/figures.jl", "tools/engine-pin.txt", "data/2012/MBodySize.csv", "data/2012/BodySize.csv", "data/2012/ChickSurvival.csv", "data/2012/FemaleSuccess.csv", "data/2012/SparrowSurvival.csv")
    mkpath(dirname(f)); isfile(f) || download("$REPO_RAW/$f", f)
end
println("engine and data ready — run the next cell for the plotting stack (several minutes; read on meanwhile)")

In [ ]:
# Cell 2 of 2 — the plotting stack. This is the slow part on a bare machine (about eight minutes measured;
# Binder pays it once at image build, so there it is seconds). Every figure in the chapter needs it.
import Pkg
Pkg.add(["Makie", "CairoMakie", "AlgebraOfGraphics"])
using CairoMakie
println("plotting ready")

---
title: "Class 9: both at once"
book: Stats Hours with Itchy
chapter: 9
type: book-chapter
status: draft
created: 2026-09-07
engines: DRM.jl 0.7.1 at 26f4c4ddc (Julia 1.10.0) · drmTMB 0.7.0 (R 4.6.0)
tags: [book, julia, GLMM, binomial, ICC, conditional-vs-marginal, DRM.jl]
deck: "Class 3 promised a repair it could not perform, because the file it was given never recorded which nest a chick came from. Here is the file that did — and the standard error the same model prints on it turns out to be answering a different question, with an interval that is wrong for either."
status_tag: Draft
status_note: "All ten classes of version 1 run end to end (1–10), with Appendix A, the preface and the coda. Every number and figure on this page was computed when the site was built; the book itself is still being written."
provenance: "Every block of Julia code on this page ran when the site was built, including the one that fails. Nothing is pasted from a session you cannot see. The one R box was run once by hand, on the date shown (2026-09-07): all four of its blocks come from a single Rscript run against the same archived file, and the box says so; the script is kept at data/ch9/ch9-r-box.R."
caveat: "The data are real. These are the Lundy Island house sparrow chicks of the author's 2012 course, read directly from the archived data/2012/SparrowSurvival.csv, which is the brood-identified version of the file Class 3 used (where the file comes from is recorded in data/2012/README.md). Nothing here is invented; the simulated quantities are drawn from a model fitted to that file, from random-number generators set up in the code you can see."
footer_note: "Stats Hours with Itchy · Class 9 of twelve rungs, ten in v1, plus a coda · draft, built 2026-09-07"
---

# Class 9: both at once

> **A note on the data, and it is the note Class 3 promised.** Class 3 fitted chick survival with
> `Binomial()` on `data/2012/ChickSurvival.csv` and said, in writing, that the chicks in a nest are
> not strangers and that the file could not be repaired, because it recorded how *big* each brood
> was and never *which* brood. Class 6 then freed the correlation between rows on
> `data/2012/BodySize.csv`, where the repeated thing was a recapture. This chapter is what happens
> when the same population is handed over with the nest written down. Same birds, same island,
> same question, one more column.

---

## Objectives

By the end of this class you should be able to:

1. Fit a generalised linear mixed model — a family *and* a grouping in the same call — and say which constant each of the two is freeing.
2. Predict, before you fit, what adding the grouping will do to a standard error, and say by how much it was wrong to leave it out.
3. Read a variance component that lives on the link scale — the scale of the model's straight-line part, before it is turned back into a probability — and turn it into an intraclass correlation, saying out loud which scale that correlation lives on and why the scale had to be invented.
4. Say why a GLMM's coefficient is a **conditional** odds ratio, why averaging over groups shrinks it towards zero, and which of the two your reader wanted.
5. Decide from the data file itself whether a second grouping is nested, crossed or neither, and whether it has enough levels to be treated as random at all.

---

## The class

**Itchy's office, 9:00 am. TOTO has brought the same chick file he brought to Class 3 and the grudge that goes with it. MOMO has read ahead and is suspicious of the word "both". EDDIE is here for the part where the two halves of the course meet. JARO, who is a statistician and turns up for the hard weeks, has been promised coffee.**

**Itchy:** Class 3 gave you the shape of the noise. Class 6 gave you the correlation between rows. Every constant either of those classes freed is still free today, and today you free both in one call. Toto, remind the room what I told you in Class 3 and could not deliver.

**Toto:** That the chicks in one nest are not independent, that my standard error was too small, and that the file could not be fixed because it never said which nest.

**Itchy:** And here is the file that says. Load it the way you already know how.

In [ ]:
#| label: setup
# tools/diagnostics.jl includes tools/figures.jl, which includes tools/theme_itchy.jl,
# so one include does all three.
include("tools/diagnostics.jl")
using DRM, DataFrames, CSV, Statistics, Random, Printf, CairoMakie
import Distributions
set_theme!(theme_itchy(:light))

# Class 6's recipe, applied without thinking about it. Watch what it does.
raw = CSV.read("data/2012/SparrowSurvival.csv", DataFrame; missingstring = "NA")

println("rows, columns: ", size(raw))
describe(raw, :eltype, :nmissing)

**Momo:** Every column is text and nothing is missing.

**Itchy:** Every column is text and nothing is missing, which is two lies for the price of one. Fit it anyway, because I want the engine to be the one who tells you.

In [ ]:
#| label: first-attempt
#| error: true
drm(bf(@formula(Survival ~ Mass2 + (1|BroodNo))), Binomial(); data = raw)

**Toto:** That is the Class 6 error again.

**Itchy:** It is the same error with a different cause, which is why you get it twice in one book. In Class 6 the file said `NA`, so we told the reader that `NA` means missing. This file, written by the same person in the same decade, leaves the cell **empty** instead. And `missingstring = "NA"` does not *add* `NA` to the list of missing markers. It **replaces** the list, and the empty string was on the list we just threw away. Name both.

In [ ]:
#| label: read-clean
raw = CSV.read("data/2012/SparrowSurvival.csv", DataFrame;
               missingstring = ["NA", ""])
println("missing values per column:")
println(describe(raw, :nmissing))

chicks = dropmissing(raw, [:Survival, :Mass2, :BroodNo])
mkpath("data/ch9")
CSV.write("data/ch9/chicks-broods.csv", chicks)

n = nrow(chicks)
broods = sort(unique(chicks.BroodNo))
J = length(broods)
class3 = CSV.read("data/2012/ChickSurvival.csv", DataFrame)

@printf("kept %d of %d rows on %d broods\n", n, nrow(raw), J)
@printf("survived: %d of %d (%.4f)\n", sum(chicks.Survival), n, mean(chicks.Survival))
@printf("Class 3's file, for comparison: %d rows, %d columns, no brood identifier\n",
        nrow(class3), ncol(class3))

**Itchy:** `{julia} n` chicks in `{julia} J` broods, once we keep the rows that have a survival, a mass and a nest. Class 3 had `{julia} nrow(class3)` rows and no way of knowing which of them shared a mother. Count the nests properly before you model them, exactly as you counted birds.

In [ ]:
#| label: per-brood
per_brood = combine(groupby(chicks, :BroodNo), nrow => :k)
ks = sort(unique(per_brood.k))

for k in ks
    @printf("%3d broods with %d chick%s\n", count(==(k), per_brood.k), k, k == 1 ? "" : "s")
end
mean_k = mean(per_brood.k)
@printf("\nmean chicks per brood: %.4f\n", mean_k)

# How much of a brood's fate is shared? Count the broods that went one way entirely.
by_brood = combine(groupby(chicks, :BroodNo), :Survival => mean => :p)
n_all_died  = count(by_brood.p .== 0)
n_all_lived = count(by_brood.p .== 1)
n_mixed     = count(0 .< by_brood.p .< 1)
n_unanimous = n_all_died + n_all_lived

@printf("\nbroods where every chick died : %d\n", n_all_died)
@printf("broods where every chick lived: %d\n", n_all_lived)
@printf("broods that went both ways    : %d\n", n_mixed)

# "You would not see that many" is a claim, so compute it rather than assert it.
# If every chick were an independent flip at the overall rate, a brood of k is
# unanimous with probability p^k + (1 - p)^k. Sum that over the broods this file has.
p_bar = mean(chicks.Survival)
exp_unanimous = sum(p_bar^k + (1 - p_bar)^k for k in per_brood.k)

@printf("\nunanimous broods, observed                    : %d\n", n_unanimous)
@printf("expected under independent flips at p = %.4f : %.1f\n", p_bar, exp_unanimous)

**Eddie:** More than half the broods went one way entirely.

**Itchy:** More than half, and that is the clustering staring at you out of the raw counts before a model has been fitted. `{julia} n_unanimous` of `{julia} J` broods had no disagreement inside them at all, and independent coin flips at the population rate would have given you about `{julia} round(Int, exp_unanimous)`. Hold the number; it comes back, and when it does it will be the best argument for a random effect you have met. (Two small things in the cell: `0 .< p .< 1` chains two comparisons, element by element, and the expected count is Class 4's generator handed to `sum`.)

**Momo:** In Class 6 you made us predict what the grouping would do to the standard error before fitting anything.

**Itchy:** And you are going to do it again, with Class 6's rule, on this file. The rule is about the **predictor**, so ask where the predictor varies.

In [ ]:
#| label: predict-the-se
# A Gaussian random-intercept model on mass itself splits mass's own variance into a
# between-brood part and a within-brood part. That split is what Class 6's rule needs.
mass_fit   = drm(bf(@formula(Mass2 ~ 1 + (1|BroodNo))), Gaussian(); data = chicks)
sd_between = re_sd(mass_fit)[:BroodNo]
sd_within  = first(sigma(mass_fit))
icc_mass   = sd_between^2 / (sd_between^2 + sd_within^2)

@printf("mass, between-brood SD : %.4f g\n", sd_between)
@printf("mass, within-brood SD  : %.4f g\n", sd_within)
@printf("mass is %.1f%% between broods and %.1f%% within them\n",
        100 * icc_mass, 100 * (1 - icc_mass))

**Momo:** Mostly within. In Class 6 tarsus was almost entirely *between* birds, and the standard error was too small. Mass is the other way round, so by that rule the standard error should not go up much.

**Itchy:** That is the rule applied correctly, and that is its honest prediction. Write it on the board. In twenty minutes the standard error will have gone up sharply, and finding out what the rule does not cover on a nonlinear link is worth more to you than a rule that was never wrong.

### The model Class 3 could fit

**Itchy:** First, the model Class 3 could fit, on today's file. Same family, same formula, no grouping. I want the comparison to be like for like, so both fits get the same `{julia} n` chicks.

In [ ]:
#| label: fit-glm
glm_fit = drm(bf(@formula(Survival ~ Mass2)), Binomial(); data = chicks)
glm_fit

**Toto:** One block, `mu`, and no σ table. That is the GLM exactly.

**Itchy:** That is Class 3 exactly, and everything it told you about it is still true. The link is still the logit, the variance is still decreed to be p(1 − p), and for a nought-or-one response that decree is arithmetic rather than a promise. Nothing about the *shape* is wrong here. What is wrong is the sentence that class warned you about and could not repair.

### Both at once

**Itchy:** One term. The same term as Class 6, in a call with a different family.

> **logit(p_ij) = β0 + β1 x_ij + u_j, with u_j ~ N(0, σ_b²) and y_ij ~ Bernoulli(p_ij)**

**Itchy:** Chick *i* in brood *j*. Read it against Class 6's line and notice what moved. β0, β1 and u_j are the same three symbols doing the same three jobs. The random effect is still normal, still centred on zero, still summarised by one spread. Two things changed: there is a **link** wrapped around the left-hand side, so the brood's nudge is a nudge in log-odds; and there is **no σ**, because the family already spent it. Momo.

**Momo:** So the random effect is on the log-odds scale and not on the scale of anything I can see.

**Itchy:** Not on the scale of anything you can see, and that is the price of the hour. Write it down now, because in fifteen minutes I am going to hand you a number that looks like a correlation and it will live on that invisible scale too.

In [ ]:
#| label: fit-glmm
glmm_fit = drm(bf(@formula(Survival ~ Mass2 + (1|BroodNo))), Binomial(); data = chicks)
glmm_fit

**Toto:** Same verb, same box. And the block at the bottom is back, but the σ table is still gone.

**Itchy:** Both of those are the point. The random-effect block is Class 6's second variance component; the absent σ is Class 3's decree. This is the first fit in the book that carries both, and the heading on the new block is the same warning it carried before. Read it, Momo.

**Momo:** "NOT the σ_b = 0 boundary."

**Itchy:** Same engine, same refusal, same reason, and Class 8 gave it its hour. Now put the two fits side by side and watch which numbers move.

In [ ]:
#| label: side-by-side
b_glm,  b_glmm  = coef(glm_fit, :mu), coef(glmm_fit, :mu)
se_glm, se_glmm = stderror(glm_fit)[1:2], stderror(glmm_fit)[1:2]

@printf("%-14s %12s %12s %12s\n", "", "no brood", "with brood", "ratio")
@printf("%-14s %12.4f %12.4f %12.4f\n", "beta0",     b_glm[1],  b_glmm[1],  b_glmm[1] / b_glm[1])
@printf("%-14s %12.4f %12.4f %12.4f\n", "beta1",     b_glm[2],  b_glmm[2],  b_glmm[2] / b_glm[2])
@printf("%-14s %12.4f %12.4f %12.4f\n", "SE(beta0)", se_glm[1], se_glmm[1], se_glmm[1] / se_glm[1])
@printf("%-14s %12.4f %12.4f %12.4f\n", "SE(beta1)", se_glm[2], se_glmm[2], se_glmm[2] / se_glm[2])
@printf("\n%-14s %12.2f %12.2f %12.2f\n", "AIC", aic(glm_fit), aic(glmm_fit),
        aic(glmm_fit) - aic(glm_fit))

se_ratio    = se_glmm[2] / se_glm[2]
slope_ratio = b_glmm[2] / b_glm[2];

**Toto:** The standard error went up by more than half.

**Itchy:** By a factor of `{julia} round(se_ratio, digits = 2)`, with Momo's prediction on the board saying it should not have. Hold that too; the reason is Jaro's, in ten minutes. And AIC fell by `{julia} round(Int, abs(aic(glmm_fit) - aic(glm_fit)))`, which is not a close call by anybody's rule of thumb, so this is not a term you can argue your way out of on grounds of parsimony.

**Toto:** In Class 3 you told me *my* interval was too narrow.

**Itchy:** I did, and it was, and no number here is Class 3's number: that fit was on a different file, with `{julia} nrow(class3)` rows and no brood column. What you have is Class 3's **model** refitted on Class 9's chicks, which is why both fits got the same `{julia} n`.

**Momo:** In Class 6 the slope barely moved and only the standard error did. This slope moved.

**Itchy:** Momo has found the difference between Class 6 and today, and it is the most important sentence of the hour, so Jaro gets ten minutes on it after the variance component. Note the size for now: the coefficient is a factor of `{julia} round(slope_ratio, digits = 2)` larger once broods are in the model. In a Gaussian mixed model that essentially does not happen. In a GLMM it happens **every time**, and it is not a bias, a bug or an improvement.

### What the brood variance says

**Itchy:** Get the two pieces out and give them the scale they live on.

In [ ]:
#| label: variance-components
sigma_b = re_sd(glmm_fit)[:BroodNo]   # brood SD, on the LOG-ODDS scale
V_brood = sigma_b^2
V_link  = pi^2 / 3                    # variance of a standard logistic distribution

icc = V_brood / (V_brood + V_link)

@printf("sigma_b, brood SD on the logit scale : %.4f\n", sigma_b)
@printf("brood variance                       : %.4f\n", V_brood)
@printf("logistic latent variance, pi^2 / 3   : %.4f\n", V_link)
@printf("latent-scale ICC                     : %.4f\n", icc)
println("\nvc(glmm_fit): ", vc(glmm_fit))

**Eddie:** Where did π² over three come from? Nothing in the data has a π in it.

**Itchy:** Nothing in the data does, and that is exactly why the number needs a label. Class 6 divided the between-bird variance by the between-bird variance plus σ², and σ was a real spread in real millimetres. Here there is no σ. The family spent it. So to get a ratio at all you have to invent a denominator, and the standard move is to imagine a **latent** continuous variable underneath the coin flip: the chick has a hidden propensity, it survives if the propensity clears a threshold, and a logit link is what you get when that hidden propensity has a standard logistic distribution. A standard logistic distribution has variance π²/3. That is the whole derivation, and Nakagawa and Schielzeth (2010) work it through properly along with the alternatives.

**Eddie:** I still cannot picture "hidden propensity" from a sentence.

**Itchy:** Then stop trying to picture the coin flip and picture what is underneath it. Same shape every time — a standard logistic curve, the one that gave you π²/3 — just slid sideways by however far that brood's own u_j pushes it. Slide it far enough and the whole curve ends up on one side of the line.

In [ ]:
#| label: fig-latent-threshold
#| fig-cap: "The logit link as a threshold on a hidden scale. Each curve is the same standard logistic density — one chick's own luck, the assumed distribution behind π²/3 — centred at a different brood's linear predictor: an average-mass chick in an average brood (middle), and the same chick in a brood pushed one and a half population standard deviations of σ_b to either side. A chick survives if its point on this scale clears the dashed threshold at zero, and the area of a curve to the right of that line is its brood's survival probability. A random effect only slides the curve; it never changes its shape — which is why a brood pushed far enough ends up almost entirely on one side of the line, the mechanism behind the unanimous broods counted at the top of this chapter."
eta_bar = mean(b_glmm[1] .+ b_glmm[2] .* chicks.Mass2)   # an average chick, fixed part only
mus      = eta_bar .+ [-1.5, 0.0, 1.5] .* sigma_b
labels_u = ["1.5 σ_b below average", "an average brood", "1.5 σ_b above average"]
zgrid    = range(minimum(mus) - 5, maximum(mus) + 5, length = 400)

fig = Figure(size = (680, 380))
ax = Axis(fig[1, 1]; xlabel = "latent propensity to survive (log-odds scale)",
    ylabel = "density", title = "a threshold on a hidden scale")
vlines!(ax, [0.0]; color = :black, linestyle = :dash, linewidth = 2,
    label = "threshold: survives above, dies below")
for (mu, lab) in zip(mus, labels_u)
    dens = Distributions.pdf.(Distributions.Logistic(mu, 1.0), zgrid)
    lines!(ax, zgrid, dens; linewidth = 2.5, label = lab)
end
Legend(fig[1, 2], ax; framevisible = false)
fig

**Momo:** So the number is a ratio of one thing we estimated to one thing we assumed.

**Itchy:** It is a ratio of one thing we estimated to one thing the **link function** assumed on your behalf, which is worse than assuming it yourself, because you were not asked. Say it out loud every time: **latent-scale**. About `{julia} string(round(Int, 100 * icc), "%")` of the variation in the latent propensity to survive is differences between broods, and the rest is a chick differing from its own siblings. Choose a **probit** link instead — the same idea as logit, a threshold on a hidden propensity, but built from the normal curve instead of the logistic one — and the denominator becomes one rather than π²/3, and the number changes without a single chick changing. It is a real quantity and it is scale-bound, and a repeatability quoted without its scale is not a repeatability, it is a rumour.

**Eddie:** Is there a version on the data scale?

**Itchy:** There is more than one, which is the honest answer and the reason I am not computing one today. Nakagawa, Johnson and Schielzeth (2017) set out the family of them and when each applies; the latent-scale one is the one that is comparable across studies with the same link, and it is the one to report first. What you may not do is compute the latent one and describe it as "the proportion of variation in survival between broods", because survival is a nought or a one and its variation is not what you just divided.

**Toto:** And can I test whether broods differ at all?

**Itchy:** You can ask, and the engine will answer and then take part of the answer back. Watch.

In [ ]:
#| label: lrt
# The test, on its own, so that what the engine says comes back on its own too.
lr  = lrtest(glm_fit, glmm_fit)
lrb = lrt_boundary(glmm_fit, glm_fit; q = 1);

In [ ]:
#| label: lrt-read
@printf("lrtest        : chi2 = %.2f on %d df, p = %.3e\n", lr.statistic, lr.dof, lr.pvalue)
@printf("lrt_boundary  : chi2 = %.2f, q = %d, p = %.3e (naive p = %.3e)\n",
        lrb.statistic, lrb.q, lrb.pvalue, lrb.pvalue_naive)

**Momo:** It warned us before it answered.

**Itchy:** It warned you that the null you asked about sits on the edge of the space the parameter lives in — a variance cannot be negative — so the ordinary chi-squared reference is the wrong one, and it named the function that uses the right one. Here the two p-values differ by a factor of two and both are far past any threshold anybody uses, so the correction changes nothing you would do. That is the *lucky* case; Class 8 was the hour where the same correction is the difference between a result and no result. Meanwhile: notice that the interesting number is not the p-value at all. It is `{julia} round(icc, digits = 2)`, and no p-value in this chapter tells you that.

### Jaro's ten minutes: whose odds ratio is it?

**Jaro:** May I have the slope back, please. Both slopes.

**Itchy:** Take them.

In [ ]:
#| label: odds-ratios
or_glmm = exp(b_glmm[2])
or_glm  = exp(b_glm[2])

@printf("GLMM slope  %.4f  ->  odds ratio %.4f\n", b_glmm[2], or_glmm)
@printf("GLM  slope  %.4f  ->  odds ratio %.4f\n", b_glm[2],  or_glm)

**Jaro:** Two odds ratios, both correct, and they answer different questions. The mixed one says: take **one brood**, hand it a chick a gram heavier, and that chick's odds of surviving are multiplied by `{julia} round(or_glmm, digits = 2)`. It is a statement about a comparison *inside* a nest, with the nest held where it is. That is a **conditional** odds ratio.

**Toto:** And the other one?

**Jaro:** The other one is closer to a statement about the population as a whole: take all the chicks at one mass and all the chicks at a mass one gram higher, average over whatever nests they happen to be in, and compare. That is a **marginal** odds ratio, and it is smaller. It is always smaller on a **logit** link, and the same holds for probit and **complementary log-log** — a third link, built from a lopsided curve rather than the symmetric ones logit and probit use. The effect has a name: non-collapsibility.

**Eddie:** Any nonlinear link, then.

**Jaro:** No — and I am glad you said it, because that is the sentence people get wrong. The **log** link, the one this book taught you in Classes 3 to 5, is collapsible for the rate ratio. Write the Poisson mean out: E[Y | x, u] = exp(β0 + β1 x + u). Averaging over u multiplies the whole thing by one constant, which moves the **intercept** and leaves β1 exactly where it was. So "nonlinear" is not the criterion. Logit is.

**Momo:** Always smaller. Prove it rather than saying it.

**Itchy:** We will not prove it, we will make the model prove it, which is this book's method. The fit believes a distribution of broods. Draw broods from it, draw chicks in those broods, fit the model that ignores broods, and see what that model recovers. The coin flip is one line: `rand(rng_marg, n) .< p` compares a uniform draw with each chick's probability, so it is true with probability `p`, and `Int.(...)` turns the trues and falses into ones and zeros for the formula. `logistic` is a one-line function of our own for the squashing curve, and the loop collects with `push!`, as before. Further down, `20_000` is twenty thousand with an underscore for the eye.

In [ ]:
#| label: marginal-by-simulation
logistic(z) = 1 / (1 + exp(-z))

bidx       = Dict(bn => i for (i, bn) in enumerate(broods))
row_brood  = [bidx[bn] for bn in chicks.BroodNo]
eta_fixed  = b_glmm[1] .+ b_glmm[2] .* chicks.Mass2   # the fixed part, u = 0

# One replicate: draw a brood effect for every brood, flip every chick's coin at
# its own probability, then fit the model that pretends broods do not exist.
rng_marg = MersenneTwister(20260907)
marg_slopes = Float64[]
for _ in 1:1000
    u = sigma_b .* randn(rng_marg, J)
    p = logistic.(eta_fixed .+ u[row_brood])
    y = Int.(rand(rng_marg, n) .< p)
    naive = drm(bf(@formula(Survival ~ Mass2)), Binomial();
                data = DataFrame(Survival = y, Mass2 = chicks.Mass2))
    push!(marg_slopes, coef(naive, :mu)[2])
end

marg_slope  = mean(marg_slopes)
marg_mcse   = std(marg_slopes) / sqrt(length(marg_slopes))
attenuation = marg_slope / b_glmm[2]

# The SPREAD of those slopes is the naive estimator's true sampling SD -- the only
# ruler that measures the naive interval against the target the naive fit is aiming at.
se_true     = std(marg_slopes)
naive_short = 1 - se_glm[2] / se_true
gap_in_se   = (b_glm[2] - marg_slope) / se_true

@printf("conditional slope (the GLMM)          : %.4f   OR %.4f\n", b_glmm[2], or_glmm)
@printf("marginal slope, %d simulated worlds : %.4f   OR %.4f   MC SE %.4f\n",
        length(marg_slopes), marg_slope, exp(marg_slope), marg_mcse)
@printf("what the real naive fit gave          : %.4f   OR %.4f\n", b_glm[2], or_glm)
@printf("\nattenuation, marginal / conditional   : %.4f\n", attenuation)
@printf("naive slope, the SE it printed        : %.4f\n", se_glm[2])
@printf("naive slope, its TRUE sampling SD     : %.4f  -> what it printed is %.1f%% too small\n",
        se_true, 100 * naive_short)
@printf("real naive fit minus simulated mean, in units of that true SD: %.2f\n", gap_in_se)

**Jaro:** There it is, and it did not need a theorem. Averaging over broods keeps `{julia} string(round(Int, 100 * attenuation), "%")` of the conditional slope and throws the rest away. And look at the third line: the slope the fixed-effects fit got on the real chicks sits `{julia} round(gap_in_se, digits = 2)` standard deviations from what the mixed model predicts that estimator would give. That is inside noise. **The naive fit was not estimating the wrong thing badly. It was estimating a different thing, roughly correctly, with a standard error that was wrong for either.**

**Momo:** You measured that gap against the last line and not against the standard error the fit printed.

**Jaro:** Because the standard error the fit printed is the thing under investigation, and a suspect ruler cannot check itself. The simulation has handed you the honest one: `{julia} round(se_true, digits = 4)`, against the `{julia} round(se_glm[2], digits = 4)` the fit reported.

**Toto:** So the fixed-effects coefficient was not a mistake.

**Itchy:** The *coefficient* was a defensible marginal estimate. The *interval* was a fiction. Those are separable and you must keep them separate, because the fix for the second is not the fix for the first. And now Momo's prediction on the board can be read properly. The standard error went up by a factor of `{julia} round(se_ratio, digits = 2)`, and that is two things at once: the coefficient changed size, because the mixed model's slope is conditional and the naive one is marginal, so its standard error changed size with it; and siblings were being treated as strangers, which is the part Class 6's rule covers — measured against its own target, the naive standard error is short by `{julia} string(round(100 * naive_short, digits = 1), "%")`, modest, exactly as a predictor that varies mostly *within* broods would lead you to expect. Momo's rule was right about the part it covers. On a nonlinear link there is a second part, and it is usually the larger one. Draw the two curves and the distinction stops being a word.

In [ ]:
#| label: fig-two-curves
#| fig-cap: "Chick survival, jittered, with two logistic curves from the same GLMM: solid for one average brood sitting at the population mean, dashed for the curve averaged over the fit's own distribution of broods. The dashed curve is flatter everywhere and never gets as close to either edge, because averaging steep curves shifted sideways from one another smooths out the steepness any one brood has."
mass_grid = range(minimum(chicks.Mass2), maximum(chicks.Mass2), length = 120)

# Conditional: a brood sitting exactly at the population average, u = 0.
p_cond = logistic.(b_glmm[1] .+ b_glmm[2] .* mass_grid)

# Marginal: average the CURVE over the distribution of broods the fit believes.
rng_curve = MersenneTwister(31415)
u_draw = sigma_b .* randn(rng_curve, 20_000)
p_marg = [mean(logistic.(b_glmm[1] + b_glmm[2] * m .+ u_draw)) for m in mass_grid]

rng_jit = MersenneTwister(77)
jit = 0.03 .* randn(rng_jit, n)

fig = Figure(size = (680, 380))
ax = Axis(fig[1, 1]; xlabel = "mass at day 2 (g)", ylabel = "probability of surviving",
    title = "one fit, two curves")
hlines!(ax, [0.0, 1.0]; color = (:grey, 0.5), linestyle = :dot)
scatter!(ax, chicks.Mass2, chicks.Survival .+ jit; markersize = 3, color = (:grey, 0.3))
lines!(ax, mass_grid, p_cond; linewidth = 2.5, label = "one average brood (u = 0)")
lines!(ax, mass_grid, p_marg; linewidth = 2.5, linestyle = :dash,
    label = "averaged over broods")
Legend(fig[1, 2], ax; framevisible = false)
fig

**Eddie:** The dashed one is flatter.

**Itchy:** Flatter everywhere, and it never gets as close to either edge. Averaging a set of steep curves that are shifted sideways from one another gives you a shallow curve, because at any mass some broods are already nearly all dead and some are nearly all alive, and neither of those is moving much. **Both lines come from the same fit and neither is a compromise.** Ask which one your sentence needs. "A gram of mass multiplies a chick's odds of surviving within its nest" wants the solid line. "Heavier chicks in this population survive at a rate of" wants the dashed one.

### The second grouping, and why it is not Year

**Momo:** The file has a `Year` column. Chicks in one year share a summer as much as chicks in one nest share a mother.

**Itchy:** They do, and that instinct is right, and the file will not let you act on it. Look at the *design* before you look at a fit.

In [ ]:
#| label: year-check
spanning = combine(groupby(chicks, :BroodNo), :Year => (x -> length(unique(x))) => :nyear)
n_years  = length(unique(chicks.Year))
n_mums   = length(unique(chicks.Mum))

@printf("broods appearing in more than one year : %d of %d\n", count(spanning.nyear .> 1), J)
@printf("distinct years                         : %d\n", n_years)
@printf("distinct mothers                       : %d\n", n_mums)

# Apply the same test to the OTHER candidate grouping, which is the one set as homework.
mums_per_brood = combine(groupby(chicks, :BroodNo), :Mum => (x -> length(unique(x))) => :nmum)
broods_per_mum = combine(groupby(chicks, :Mum), :BroodNo => (x -> length(unique(x))) => :nb)
n_split_broods = count(mums_per_brood.nmum .> 1)

@printf("broods carrying more than one Mum      : %d of %d\n", n_split_broods, J)
@printf("broods per mother                      : %d to %d, median %.0f\n",
        minimum(broods_per_mum.nb), maximum(broods_per_mum.nb), median(broods_per_mum.nb))
println()
combine(groupby(chicks, :Year),
        nrow => :chicks,
        :BroodNo => (x -> length(unique(x))) => :broods,
        :Survival => mean => :survived)

**Itchy:** Read the first line. **No brood appears in two years**, so `BroodNo` is not *crossed* with `Year`, it is **nested inside** it. A crossed design is one where the same brood turns up in several years and the same year holds several broods; you have half of that and only half. And read the second line: `{julia} n_years` years. Eddie, you have fitted things.

**Eddie:** You cannot estimate the spread of four numbers.

**Itchy:** You cannot usefully estimate the spread of `{julia} n_years` numbers, no. Software will often fit it and print a standard deviation, and the standard deviation of four draws is not a quantity you would report. Bolker et al. (2009) put a working floor at around five or six levels and say plainly what to do below it: put the factor in the mean model as a fixed effect and stop pretending you are estimating a distribution. So do that.

In [ ]:
#| label: fit-year-fixed
chicks.YearF = string.(chicks.Year)   # a factor must be text, or the formula treats it as a number
year_fixed = drm(bf(@formula(Survival ~ Mass2 + YearF + (1|BroodNo))), Binomial(); data = chicks)
year_fixed

**Itchy:** Three more rows in the `mu` table, one per year beyond the first, and nobody had to estimate a distribution from four points. If you want a second *random* grouping in this file, there is a candidate with `{julia} n_mums` levels in the `Mum` column — and two things stand between you and it. One is the engine: on a binomial response it will take a second grouping beside the brood, converge, and hand back an infinite standard error for every level without a word of warning, which is worse than a refusal; [Where the engine stops](appendix-b-engine.html) records it. The other is the design, which comes first whatever the engine can do. `{julia} n_split_broods` of the `{julia} J` broods carry more than one mother, and a mother holds up to `{julia} maximum(broods_per_mum.nb)` broods. So `Mum` is not nested above `BroodNo` and it is not cleanly crossed with it either: it is **partially crossed**, and the honest answer to "which is it" is that neither word fits. That is question six.

### What a random effect is, when you cannot see it

**Eddie:** In Class 6 you answered "what is a random intercept" with the shrinkage picture. Do it again here.

**Itchy:** The engine will not draw it for me: `ranef` comes back empty on a binomial fit, silently ([Where the engine stops](appendix-b-engine.html) records it), so the per-brood effects come from `binomial_group_modes` in `tools/diagnostics.jl`. What it computes is one sentence. A brood's effect is the value of u that makes that brood's own chicks most likely **and** pays the penalty for sitting far from zero: the data pulling one way, the population pulling back. Switch the penalty off and you get what the brood says on its own evidence, borrowing nothing. Those are the diamonds and the circles of Class 6's picture, on the log-odds scale, and I want both.

In [ ]:
#| label: brood-effects
pulled = binomial_group_modes(chicks.Survival, eta_fixed, chicks.BroodNo, sigma_b)
own    = binomial_group_modes(chicks.Survival, eta_fixed, chicks.BroodNo, sigma_b;
                              penalised = false)

mixed    = isfinite.(own.u)        # broods with a finite estimate on their own evidence
n_no_own = count(isinf, own.u)     # broods with none: the maximiser is at infinity

@printf("largest |gradient| at the answer, over %d broods  : %.1e\n", J, maximum(abs, pulled.grad))
@printf("broods with a finite estimate on their own evidence: %d of %d\n", count(mixed), J)
@printf("broods with no finite estimate of their own        : %d of %d\n", n_no_own, J)
@printf("all %d finite once the population pulls back       : %s\n", J, all(isfinite, pulled.u))

# How hard the population pulls a unanimous nest depends on how many chicks it had.
k_of   = [count(==(g), chicks.BroodNo) for g in pulled.levels]
k_big  = maximum(k_of[.!mixed])    # the largest nest that went one way entirely
pull_1 = median(abs.(pulled.u[.!mixed .& (k_of .== 1)]))
pull_k = median(abs.(pulled.u[.!mixed .& (k_of .== k_big)]))
@printf("\nunanimous nest of %d chick : median |effect kept| %.2f\n", 1, pull_1)
@printf("unanimous nest of %d chicks: median |effect kept| %.2f\n", k_big, pull_k)

In [ ]:
#| label: fig-shrinkage
#| fig-cap: "Sixteen of the mixed broods — at least one chick lived and at least one died, so a finite own-evidence estimate exists to draw — at even steps through their ordering, on the log-odds scale. The helper is Class 6's, so its legend uses Class 6's words: here a circle (\"raw mean\") is the brood's effect on its own evidence, with the penalty switched off, and a diamond (\"BLUP\") is the value the fit keeps once the population pulls back. Every diamond sits closer to zero than its circle."
# Broods at even steps through the ordering, so the choice is a rule and not a hand.
idx   = findall(mixed)
order = idx[sortperm(own.u[idx])]
shown = order[round.(Int, range(1, length(order), length = 16))]
kept  = pulled.u[shown] ./ own.u[shown]

@printf("fraction of its own signal a brood keeps, among those drawn: %.2f to %.2f\n",
        minimum(kept), maximum(kept))
fig_shrinkage(pulled.levels[shown], own.u[shown], pulled.u[shown])

**Eddie:** Every diamond is closer to zero than its circle, same as Class 6.

**Itchy:** Same picture, and the fraction each brood keeps runs from `{julia} round(minimum(kept), digits = 2)` to `{julia} round(maximum(kept), digits = 2)` among the broods drawn. But the picture is the smaller half of the answer, because it could only be drawn for broods that had a circle. Read the cell above it. On `{julia} n_no_own` of these `{julia} J` broods **the circle does not exist** — and that is the number I told you to hold: the `{julia} n_unanimous` unanimous broods you counted from the raw file before a model was fitted, every chick dead or every chick alive. Toto, what does a fixed effect for one of those broods look like?

**Toto:** Very negative, or very positive.

**Itchy:** Not very. **Infinite.** A nest whose chicks all died has a likelihood that keeps climbing as that nest's own effect goes to minus infinity: there is no best value, only "further". A model with a fixed effect per brood has no finite estimate for more than half the nests in this file. Now, what did the mixed model give those broods?

**Toto:** Something finite, because they are in the fit.

**Itchy:** Something finite, and something *different* for each of them, because the population pulls every brood back towards zero and pulls a unanimous nest of one harder than a unanimous nest of `{julia} k_big`: the last two lines of the cell put the lone chick's nest at `{julia} round(pull_1, digits = 2)` on the log-odds scale and the full nest at `{julia} round(pull_k, digits = 2)`. That is not a technicality; it is the strongest argument for random effects anybody will ever show you. Class 6 said a random intercept is the decision to treat groups as draws from one population. Today that decision is the difference between an estimate and no estimate at all.

**Momo:** And is it right, or merely finite?

**Itchy:** It is right in the sense that it is the value the model considers most likely for that nest, given everything the model believes. Whether the *model* is right is the next section.

### Is the model any good?

**Itchy:** Quantile residuals, with a seed, as in Class 3. And then a trap that class did not have.

In [ ]:
#| label: fig-diagnostic
#| fig-cap: "Worm plot of the GLMM's randomised quantile residuals, at randomisation seed 909, judged against a brood sitting at the population average. The body of the worm sits above the zero line while both tails dip back below it, and the spread printed above the plot is above one."
qres = residuals(glmm_fit; type = :quantile, rng = MersenneTwister(909))

@printf("quantile residuals: mean %.4f, SD %.4f\n", mean(qres), std(qres))

fig_diagnostic(qres; title = "GLMM, randomised quantile residuals")

**Toto:** The spread is above one.

**Itchy:** Above one — compared with what? A quantile residual is standard normal when each observation is compared with **its own** correct distribution, and for a chick that means its own probability of surviving. So before you read the spread as a verdict, ask which probability each chick was judged against. Class 6 made you ask exactly this about a residual, and the answer was marginal-or-conditional. Same question, one level harder.

In [ ]:
#| label: qres-reference
# A randomised quantile residual for a 0/1 response is Phi^-1 of
#   u = 1 - p + p*U   if y = 1,      u = (1 - p)*U   if y = 0,   U ~ Uniform(0,1).
# The only free choice is WHICH p. Reproduce the engine first, so the recipe is verified.
Phi_inv(u) = Distributions.quantile(Distributions.Normal(), u)

function qresid(y, p, rng)
    U = rand(rng, length(p))
    pit = [yi == 1 ? (1 - pi_) + pi_ * ui : (1 - pi_) * ui
           for (yi, pi_, ui) in zip(y, p, U)]
    Phi_inv.(clamp.(pit, 1e-12, 1 - 1e-12))
end

p_cond_row = fitted(glmm_fit)                                    # u = 0: a typical brood
p_marg_row = [mean(logistic.(e .+ u_draw)) for e in eta_fixed]   # averaged over broods

r_cond = qresid(chicks.Survival, p_cond_row, MersenneTwister(909))
r_marg = qresid(chicks.Survival, p_marg_row, MersenneTwister(909))

@printf("engine                      : SD %.4f\n", std(qres))
@printf("by hand, p from fitted()    : SD %.4f\n", std(r_cond))
@printf("by hand, p averaged over u  : SD %.4f\n", std(r_marg))
@printf("\nmean fitted probability, u = 0 : %.4f\n", mean(p_cond_row))
@printf("mean probability over broods   : %.4f\n", mean(p_marg_row))
@printf("chicks that actually survived  : %.4f\n", mean(chicks.Survival))

In [ ]:
#| label: fig-diagnostic-marginal
#| fig-cap: "The same chicks' quantile residuals, recomputed against probabilities averaged over the fit's own distribution of broods instead of a single average brood. The body of the worm now lies close to the zero line, though both tails still dip below it, and its spread, printed above, is close to one: the first plot's excess was the reference probability, not the model."
fig_diagnostic(r_marg; title = "same chicks, marginal reference")

**Itchy:** Line two reproduces line one exactly, so the recipe is not a guess. Line three changes one thing — the probability each chick is judged against is averaged over the broods the model believes in, instead of being the probability for a brood sitting at the average — and the spread comes down from `{julia} round(std(r_cond), digits = 3)` to `{julia} round(std(r_marg), digits = 3)`. Look at the second worm plot and then at the first. Momo, say what that means.

**Momo:** The residuals were not too wide. The yardstick was too narrow.

**Itchy:** The yardstick was the wrong one, which is a different diagnosis with a different action. `fitted` sets every brood's own effect to zero. In Class 6 that was the same as averaging over birds, because the model was a straight line and the average of a straight line is the line at the average. Here there is a link function in the way, and a brood set to zero is *not* the average over broods: read the three probabilities at the bottom of the cell. A chick is not, on average, a chick from an average brood.

**Toto:** So is the model good or not?

**Itchy:** What you may write is what this shows: against the reference the model itself implies, the per-chick residuals land about where a correct model would put them. Two cautions, both cheap. That spread is one draw of a randomisation, so report the seed. And this check looks at each chick's residual **on its own**, and the thing we spent the hour on is that chicks in a nest are not on their own; a per-observation check cannot see a correlation it has already averaged out. How to simulate a diagnostic's own null — fit the model to worlds it invented, and count what the check says there — is [Class 5a](wk5-diagnostics.html)'s, written out there as a loop over a thousand invented worlds; it is not repeated this hour.

### One box, and it is earned

**Itchy:** Last thing, and it is the reason Momo will get an email from a collaborator. Somebody will fit this in R, and they will get a different brood standard deviation from ours. A GLMM's likelihood contains an integral, one per brood, over that brood's unobserved effect, and for a binomial response it has no closed form; every package approximates it, and the printed variance component depends on which approximation. Ours spreads a fixed grid of thirty-two points around zero and adds up. That is not the only rule in the room.

<!-- box: translate | id: glmm-quadrature | ch: 09 | checked: 2026-09-07 -->

> **↔ TRANSLATE: the same GLMM in R, and a disagreement that is not about R**
>
> **Three** rules meet on this page:
>
> - **Laplace** — one quadratic approximation at the mode. Fast, and the default in both
>   `lme4::glmer` (`nAGQ = 1`) and `drmTMB`.
> - **Non-adaptive Gauss–Hermite** — a fixed grid of thirty-two nodes centred at **zero**,
>   the expected value of the random effect. This is our engine's binomial `(1|g)` path, and
>   it produced every Julia number in this chapter; the appendix
>   [Where the engine stops](appendix-b-engine.html) records what it does and does not offer.
> - **Adaptive Gauss–Hermite** — the same quadrature, but re-centred and re-scaled at each
>   group's **conditional mode**, the value of that group's own effect the model finds most
>   likely. `glmer` with `nAGQ > 1`. Pinheiro and Bates (1995) is where the rules are compared.
>
> Read the four blocks below against the Julia output above. Laplace and one-point quadrature
> give the same brood SD, well away from the other two. Our number and `nAGQ = 25` are **two
> different approximations that agree**, not one integral computed twice. And read `nAGQ = 9`
> against `nAGQ = 25`: the log-likelihood at nine is slightly **higher**, and twenty-five is the
> converged one. Quadrature error is not signed, so "higher log-likelihood means closer to the
> integral" is not a rule; the evidence that you have arrived is that two different rules agree
> and that pushing one of them further stops moving it. Name the approximation with the number.
>
> The R block below was run once by hand, on the date shown: all four blocks come from one
> `Rscript` run on 2026-09-07 against the same archived file this chapter reads, and none of
> it re-runs when the page is built. The script is kept at `data/ch9/ch9-r-box.R`.

```r
# run once by hand on 2026-09-07; this block does not re-run when the page is built.
# All four blocks produced together by a single run of:
#   Rscript data/ch9/ch9-r-box.R
#   R 4.6.0, drmTMB 0.7.0, lme4 2.0.1, reading data/2012/SparrowSurvival.csv and
#   dropping rows missing Survival, Mass2 or BroodNo (1600 rows, 484 broods)

--- drmTMB (Laplace) ---
                 estimate std_error
mu:(Intercept) -3.9153478 0.3263290
mu:Mass2        0.9341965 0.0797836
                    estimate std_error
sd:mu:(1 | BroodNo)  1.70008 0.1509664
logLik: -912.4

--- lme4::glmer, nAGQ = 1 (Laplace, the default) ---
            Estimate Std. Error  z value
(Intercept)  -3.9152     0.3262 -12.0028
Mass2         0.9342     0.0798  11.7127
  brood SD = 1.700013   logLik = -912.3729611   AIC = 1830.745922

--- lme4::glmer, nAGQ = 9 (adaptive GHQ) ---
            Estimate Std. Error  z value
(Intercept)  -3.9493     0.3328 -11.8670
Mass2         0.9401     0.0813  11.5671
  brood SD = 1.861659   logLik = -903.2083452   AIC = 1812.41669

--- lme4::glmer, nAGQ = 25 (adaptive GHQ) ---
            Estimate Std. Error  z value
(Intercept)  -3.9485     0.3326 -11.8704
Mass2         0.9399     0.0812  11.5691
  brood SD = 1.860915   logLik = -903.2165109   AIC = 1812.433022
```

**Momo:** The last block is our fit.

**Itchy:** The last block **agrees with** our fit, to four decimals on the brood standard deviation and on both slopes, and to two on the AIC. It is not our fit: ours prints a log-likelihood of `{julia} round(loglik(glmm_fit), digits = 4)` and that block prints `-903.2165109`. Two rules, four decimals of agreement, and no further.

**Toto:** And the first two blocks?

**Itchy:** Also correct, and they are `glmer`'s default and `drmTMB`'s default, and neither of them asked you. Toto, what is the size of the thing that was hiding in a default?

**Toto:** The brood standard deviation.

**Itchy:** Which is the number the whole hour was about, and the numerator of the intraclass correlation, and the thing you were going to call repeatability in a paper. Laplace says `1.700`, from the box. The other two say `{julia} round(sigma_b, digits = 4)` — ours, computed above — and `glmer`'s `1.860915`, from the box. That is not a rounding difference and it is not a language difference — it is how much arithmetic each package spent on the same integral, chosen for you at install time. **This is Class 8 with a different mechanism.** There the default you never chose was ML against REML. Here it is which approximation your package picked. Same sentence fixes both: name it, in the methods, with the number.

**Toto:** So today's summary is: use a GLMM.

**Itchy:** Today's summary is that "GLMM" is two decisions wearing one word, that the coefficient it gives you answers a narrower question than the one it replaced, and that everything Class 6 taught about groups is still true on a scale you cannot see.

---

## Summary

### Stats stuff

- **A GLMM frees two constants at once.** The family decrees the link and the variance function; the grouping term frees the correlation between rows. Neither repair changes what the other does, and the call is one line.
- **The standard error was the thing most wrong, and it was wrong in two separable ways.** On this file the brood term multiplied the slope's standard error by `{julia} round(se_ratio, digits = 2)`. Part of that is the coefficient changing size, because the mixed slope is conditional and the naive one marginal; part is siblings treated as strangers, and that part — the naive standard error against its own target, by simulation — is a shortfall of `{julia} string(round(100 * naive_short, digits = 1), "%")`. The where-does-the-predictor-vary rule covers the second part and not the first.
- **σ_b lives on the link scale.** For a logit link the brood standard deviation is in log-odds, and there is no σ to divide it by, because the family spent it. To get a ratio you must supply a denominator, and the usual one is **latent**: the standard logistic distribution's variance, π²/3. Report the resulting ICC as **latent-scale**, always (Nakagawa & Schielzeth 2010). Switch to a probit link and the denominator becomes 1 and the number changes with no change to any bird. Data-scale alternatives exist and are not the same quantity (Nakagawa, Johnson & Schielzeth 2017).
- **Conditional and marginal odds ratios are different quantities, and both are correct.** The GLMM coefficient is **conditional**: it compares two chicks in the same brood. Averaging the fitted curve over the distribution of broods gives a **marginal** curve that is flatter, and a marginal odds ratio closer to one. This is non-collapsibility, and it is not bias. It holds for logit, probit and complementary log-log, not for the log link, which is collapsible for the rate ratio. On this file the marginal slope keeps `{julia} string(round(Int, 100 * attenuation), "%")` of the conditional one; the naive fit was estimating a different, defensible quantity, with an interval that was wrong for either.
- **A likelihood-ratio test for a variance component stands on a boundary.** The engine says so in a warning and names the corrected entry point. Here the correction changes nothing anybody would act on; Class 8 is where it changes everything. The interesting number in this section was never the p-value.
- **Levels, not significance, decide whether a grouping can be random.** Bolker et al. (2009) put the working floor near five or six levels and say to use a fixed effect below it; `Year` has `{julia} n_years`, so it went into the mean model.
- **Check nesting from the data, and be willing to answer "neither".** No brood in this file spans two years, so brood is nested inside year rather than crossed with it. `Mum` is not so tidy: `{julia} n_split_broods` of the `{julia} J` broods carry more than one mother, and a mother holds up to `{julia} maximum(broods_per_mum.nb)` broods, which makes the two **partially crossed**. Counting in the file is what settles this; the formula never will.
- **Shrinkage is not a refinement on a binary response; it is often the only estimate there is.** `{julia} n_unanimous` of the `{julia} J` broods in this file are unanimous — every chick died, or every chick lived — and for those a fixed effect per brood has no finite estimate. The mixed model returns a finite, different number for each of them, pulled less the more chicks the nest had: a median of `{julia} round(pull_1, digits = 2)` on the log-odds scale for a unanimous nest of one, `{julia} round(pull_k, digits = 2)` for a unanimous nest of `{julia} k_big`. That is the population supplying what the nest could not, and the shrinkage figure is the same picture on the broods that had an estimate of their own.
- **Ask which distribution a residual is judged against.** Randomised quantile residuals are standard normal under a correct model only if each observation is compared with its own correct probability. `fitted` on a GLMM sets every group effect to zero, and on a nonlinear link that is *not* the average over groups: judged against the u = 0 probabilities the spread was `{julia} round(std(r_cond), digits = 3)`, and against the probability averaged over broods it was `{julia} round(std(r_marg), digits = 3)`. A per-observation check cannot see the correlation the model was fitted to handle.
- **The likelihood has an integral in it, and packages approximate it differently.** `glmer` and `drmTMB` default to **Laplace** (brood SD 1.700, from the box); ours is a fixed thirty-two-node Gauss–Hermite grid (`{julia} round(sigma_b, digits = 4)`); `glmer` at `nAGQ > 1` re-centres that grid at each group's own mode (1.860915 from the box). Name the approximation next to the number, as you name the estimator.

### Julia you used

- **`Int.(rand(rng, n) .< p)`.** A Bernoulli draw per row: a uniform compared with a probability gives trues and falses, and `Int.` makes them ones and zeros.
- **`0 .< p .< 1`.** Two comparisons chained, element by element.
- **`logistic(z) = 1 / (1 + exp(-z))`.** A one-line function of your own for a curve the chapter uses repeatedly; `logistic.(v)` applies it to a vector.
- **`20_000`.** An underscore inside a number is ignored; it is there for the reader.
- **`[... for (yi, pi_, ui) in zip(y, p, U)]`.** `zip` over three vectors at once, each triple unpacked into three names, with a one-line `if` inside.
- **`isfinite.(v)`, `count(isinf, v)`.** A mask of the finite entries, and a count of the infinite ones; `Inf` is a value a vector can hold, and here it is the honest answer for a brood with no finite estimate.
- **`clamp.(v, lo, hi)`.** Pulls every element inside a range, here to keep a quantile finite.
- **`string.(df.Year)`.** Converts a numeric column to text. A grouping or factor stored as a number must be converted, or the formula treats it as a covariate.
- **`:Year => (x -> length(unique(x))) => :nyear`.** An anonymous function inside `combine`, here counting distinct values per group.

### Calls you used

- `CSV.read(path, DataFrame; missingstring = ["NA", ""])`: `missingstring` **replaces** the list of missing markers rather than adding to it, and the empty string is on the default list. A file using both conventions needs both spelled out.
- `drm(bf(@formula(y ~ x + (1|g))), Binomial(); data = df)`: the whole of a GLMM: a family and a grouping term in one call.
- `re_sd(fit)[:g]`, `vc(fit)`: the group SD, here on the **log-odds** scale, and the variance. There is no `sigma(fit)` on a binomial fit; the family has spent it.
- `lrtest(reduced, full)` and `lrt_boundary(full, reduced; q = 1)`: the naive test, which warns when the added block is a variance component, and the chi-bar-squared one it points at.
- `binomial_group_modes(y, eta, group, sigma_b; penalised = true)` from `tools/diagnostics.jl`: the per-group effects the engine's `ranef` does not yet return on a binomial fit, as `(; levels, u, grad)`; with `penalised = false`, what each group says on its own, `±Inf` where that has no finite answer.
- `fitted(fit)` on a mixed fit: the prediction with every group effect **set to zero**, which on a nonlinear link is not the average over groups. Average the curve yourself if you want marginal.
- `residuals(fit; type = :quantile, rng = MersenneTwister(seed))`: seeded; check which probability it judges each observation against before reading the spread.
- **Seeding.** Every random draw on this page comes from an explicit `MersenneTwister` handed to the function that needs it. Never the global generator, and never `Random.seed!`.
- **The simulation thread.** One cell, and it is the trap Class 6 met, on a new family.

In [ ]:
#| label: simulate-refit
rng_sim = MersenneTwister(20260907)
ysim = simulate(glmm_fit; nsim = 1, rng = rng_sim)

one_rep = DataFrame(Survival = Int.(ysim[:, 1]), Mass2 = chicks.Mass2,
                    BroodNo = chicks.BroodNo)
zero_refit = drm(bf(@formula(Survival ~ Mass2 + (1|BroodNo))), Binomial(); data = one_rep);

In [ ]:
#| label: simulate-read
@printf("responses simulate() produced   : %s\n", sort(unique(ysim)))
@printf("brood SD refitted from that draw: %.4f\n", re_sd(zero_refit)[:BroodNo])
@printf("brood SD in the fit it came from: %.4f\n", sigma_b)

`simulate` gives back nought-and-one responses, because it draws from the family the fit was
given; and the refit puts the brood standard deviation on the boundary at zero, because — as Class 6
found — `simulate` draws with every group effect set to zero. To ask whether the brood variance is
recoverable, draw the brood effects yourself, as the marginal simulation above did, and refit the
mixed model to each draw. [Appendix A](appendix-a-simulation.html) shows the shape of such a study —
refit, collect, count — on a slope with no random effects; a recovery study of a variance component
is not in this book.

---

## Further reading

*Graded by depth. Checked on 2026-09-07 against OpenAlex, an open catalogue of research papers.*

1. **Bolker, B. M., Brooks, M. E., Clark, C. J., Geange, S. W., Poulsen, J. R., Stevens, M. H. H. & White, J.-S. S. (2009) "Generalized linear mixed models: a practical guide for ecology and evolution", *Trends in Ecology & Evolution* 24:127–135.** doi:10.1016/j.tree.2008.10.008. Start here: the paper that put this chapter in front of biologists, including the estimation choices that make packages disagree and how many levels a random effect needs.
2. **Nakagawa, S. & Schielzeth, H. (2010) "Repeatability for Gaussian and non-Gaussian data: a practical guide for biologists", *Biological Reviews* 85:935–956.** doi:10.1111/j.1469-185X.2010.00141.x. Class 6's source, and the reason this chapter's ICC has the word "latent" attached to it: where π²/3 comes from and what the alternatives are.
3. **Harrison, X. A., Donaldson, L., Correa-Cano, M. E., Evans, J., Fisher, D. N., Goodwin, C. E. D., Robinson, B. S., Hodgson, D. J. & Inger, R. (2018) "A brief introduction to mixed effects modelling and multi-model inference in ecology", *PeerJ* 6:e4794.** doi:10.7717/peerj.4794. The practical companion to item 1: small numbers of levels, singular fits, and the model-selection habits that go wrong around random effects.
4. **Nakagawa, S., Johnson, P. C. D. & Schielzeth, H. (2017) "The coefficient of determination R² and intra-class correlation coefficient from generalized linear mixed-effects models revisited and expanded", *Journal of the Royal Society Interface* 14:20170213.** doi:10.1098/rsif.2017.0213. Where the latent-scale ratio sits among the alternatives, and how to get a data-scale version when that is the quantity you want.
5. **Zuur, A. F., Ieno, E. N., Walker, N. J., Saveliev, A. A. & Smith, G. M. (2009) *Mixed Effects Models and Extensions in Ecology with R*, Springer.** doi:10.1007/978-0-387-87458-6. The GLMM chapters are written for biologists with awkward nested data, and take the time on what a random effect buys that this chapter had to compress.
6. **Pinheiro, J. C. & Bates, D. M. (1995) "Approximations to the log-likelihood function in the nonlinear mixed-effects model", *Journal of Computational and Graphical Statistics* 4:12–35.** doi:10.1080/10618600.1995.10474663. The source for the box: what Laplace and the two Gauss–Hermite rules do to the integral, and why the answers differ by an amount that depends on the data.
7. **Dunn, P. K. & Smyth, G. K. (1996) "Randomized quantile residuals", *Journal of Computational and Graphical Statistics* 5:236–244.** doi:10.1080/10618600.1996.10474708. Class 3's diagnostic; reread it asking "which distribution is F_i?", because in a mixed model that is a choice and not a given.

---

## Exercises

In [ ]:
#| label: exercise-checks
#| echo: false
#| output: false
# Numbers the exercise checks quote, computed from the named files.
ex_wrong = CSV.read("data/2012/SparrowSurvival.csv", DataFrame; missingstring = "NA")
ex_right = CSV.read("data/2012/SparrowSurvival.csv", DataFrame; missingstring = ["NA", ""])
ex_n_wrong = nrow(dropmissing(ex_wrong, [:Survival, :Mass2, :BroodNo]))
ex_n_right = nrow(dropmissing(ex_right, [:Survival, :Mass2, :BroodNo]))
ex_ch = dropmissing(CSV.read("data/ch9/chicks-broods.csv", DataFrame), [:JulianDate])
ex_glm = drm(bf(@formula(Survival ~ JulianDate)), Binomial(); data = ex_ch)
ex_glmm = drm(bf(@formula(Survival ~ JulianDate + (1|BroodNo))), Binomial(); data = ex_ch)
ex_se_ratio = stderror(ex_glmm)[2] / stderror(ex_glm)[2]
ex_slope_ratio = coef(ex_glmm, :mu)[2] / coef(ex_glm, :mu)[2]
ex_date_fit = drm(bf(@formula(JulianDate ~ 1 + (1|BroodNo))), Gaussian(); data = ex_ch)
ex_date_within = 1 - repeatability(ex_date_fit).estimate
ex_sb = re_sd(ex_glmm)[:BroodNo]
ex_icc = ex_sb^2 / (ex_sb^2 + pi^2 / 3)
ex_or10 = exp(10 * coef(ex_glmm, :mu)[2])
ex_broods = sort(unique(ex_ch.BroodNo))
ex_bidx = Dict(bn => i for (i, bn) in enumerate(ex_broods))
ex_row_brood = [ex_bidx[bn] for bn in ex_ch.BroodNo]
ex_eta = coef(ex_glmm, :mu)[1] .+ coef(ex_glmm, :mu)[2] .* ex_ch.JulianDate
ex_rng = MersenneTwister(1)
ex_marg = Float64[]
for _ in 1:1000
    u = ex_sb .* randn(ex_rng, length(ex_broods))
    p = logistic.(ex_eta .+ u[ex_row_brood])
    y = Int.(rand(ex_rng, nrow(ex_ch)) .< p)
    push!(ex_marg, coef(drm(bf(@formula(Survival ~ JulianDate)), Binomial();
                            data = DataFrame(Survival = y, JulianDate = ex_ch.JulianDate)), :mu)[2])
end
ex_or10_marg = exp(10 * mean(ex_marg))
ex_ch.YearF = string.(ex_ch.Year)
ex_year_re = drm(bf(@formula(Mass2 ~ JulianDate + (1|BroodNo) + (1|YearF))), Gaussian(); data = ex_ch)
ex_sd_year = re_sd(ex_year_re)[:YearF]
ex_by_brood = combine(groupby(ex_ch, :BroodNo), :Survival => mean => :p)
ex_unanimous = count((ex_by_brood.p .== 0) .| (ex_by_brood.p .== 1))

Graded by depth: the first three take ten minutes each. Every exercise names a file under `data/` that exists; do it on your own organism as well where you have one. A *check* is a number computed from that file when this page was built — match it before going on. **For every question, paste your code and then explain in your own words what each line does**, as if to somebody who has done Classes 3 and 6 and not Class 9.

1. **Break the file, then find the grouping.** Read `data/2012/SparrowSurvival.csv` once with `missingstring = "NA"` and once with `missingstring = ["NA", ""]`, drop the rows missing `Survival`, `Mass2` or `BroodNo` from each, and report how many rows each version keeps and what type `Survival` has in each. One sentence on what would have happened if you had never checked. *Check:* `{julia} ex_n_wrong` rows and `{julia} ex_n_right` rows; in the first version `Survival` is text.

2. **Both fits, both standard errors, one dataset.** On `data/ch9/chicks-broods.csv`, with the rows missing `JulianDate` dropped, fit `Survival ~ JulianDate` with and without `(1 | BroodNo)`. Before you look, predict the direction of the standard-error change from how much hatching date varies within broods versus between them — **compute** that split with a Gaussian random-intercept model on `JulianDate` itself, do not eyeball it. Report both slopes, both standard errors and the two ratios. Afterwards say whether you were right, and which part of the change your prediction could not have covered. *Check:* the within-brood share of `JulianDate` is `{julia} round(ex_date_within, digits = 2)`; the standard error changes by a factor of `{julia} round(ex_se_ratio, digits = 2)` and the slope by `{julia} round(ex_slope_ratio, digits = 2)`.

3. **Name the scale.** Compute the latent-scale ICC from `re_sd(fit)` and π²/3 for the mixed fit of question 2. Then write the single sentence you would put in a paper, containing the word "latent" and the name of your link function, and say in one sentence what would change in that sentence, and what would not, under a probit link. *Check:* σ_b = `{julia} round(ex_sb, digits = 2)` on the log-odds scale, ICC = `{julia} round(ex_icc, digits = 2)`.

4. **Conditional or marginal?** Report the odds ratio per ten days of hatching date from the mixed fit. Then adapt the chapter's thousand-worlds loop to this fit, with `MersenneTwister(1)`, to get the marginal slope, and report the marginal odds ratio per ten days beside the conditional one. Two sentences: which of the two a reader comparing early and late broods across the whole island wants, and which one compares two chicks in the same nest. *Check:* conditional `{julia} round(ex_or10, digits = 2)`, marginal `{julia} round(ex_or10_marg, digits = 2)`.

5. **Too few levels.** The file's `Year` has `{julia} length(unique(ex_ch.Year))` levels. Convert it to text with `string.(df.Year)`, add `(1 | YearF)` to a Gaussian model of `Mass2 ~ JulianDate + (1 | BroodNo)`, and report the year standard deviation it prints. Then refit with `YearF` as a fixed effect instead. In three sentences: what did the random version claim to estimate, why would you not report it, and what does the fixed version report in its place? *Check:* the year SD is `{julia} round(ex_sd_year, digits = 3)` g, from four numbers.

6. **Count the design before you trust the formula.** `data/2012/SparrowSurvival.csv` carries a `Mum` column with `{julia} n_mums` levels. Count how many of the `{julia} J` broods carry more than one `Mum`, and how many broods a mother holds; the chapter makes it `{julia} n_split_broods` and up to `{julia} maximum(broods_per_mum.nb)`. Say which of "nested", "crossed" and "neither" describes `Mum` against `BroodNo`, and why the formula could never have told you. Then fit `(1|BroodNo)` and `(1|Mum)` **separately** on the same rows and say which grouping the chicks share more of their fate through.

7. **The broods with no answer.** Count the broods in `data/ch9/chicks-broods.csv` in which survival is unanimous, using `combine(groupby(df, :BroodNo), :Survival => mean => :p)`. Report the number and the fraction. Then explain in two sentences why a fixed-effect-per-brood model cannot give those broods a finite estimate, and what the mixed model used instead. *Check:* `{julia} ex_unanimous` of `{julia} nrow(ex_by_brood)` broods.

**If you have R.** Fit the chapter's model with `glmer` at `nAGQ = 1`, `nAGQ = 9` and `nAGQ = 25`, and report all three group standard deviations and all three log-likelihoods. Then write the methods sentence you would use, containing both the estimator and the integral approximation. In the chapter's box the log-likelihood at nine is higher than at twenty-five, and twenty-five is the converged answer, so "believe the higher log-likelihood" is not available to you: say in two sentences what evidence you would use instead.